# Grok-robotics-06-classical (hardened)

**Stage 06 — Classical robot control**

## Fix notes
Improve PD tracking with slower reference, velocity feedforward, and tuned gains.


In [ ]:

import json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
gpu={"cuda":False,"device_count":0,"names":[]}
try:
    import torch
    gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
except Exception as e:
    gpu["error"]=str(e)
print(gpu)

class TwoLinkArm:
    def __init__(self, l1=1.0, l2=1.0, dt=0.01, damping=0.05):
        self.l1,self.l2,self.dt,self.damping=l1,l2,dt,damping
        self.reset()
    def reset(self,q=None):
        self.q=np.array([0.4,-0.3],float) if q is None else np.array(q,float)
        self.dq=np.zeros(2); return self.q.copy()
    def fk(self,q=None):
        q=self.q if q is None else q
        return np.array([self.l1*np.cos(q[0])+self.l2*np.cos(q[0]+q[1]), self.l1*np.sin(q[0])+self.l2*np.sin(q[0]+q[1])])
    def jac(self,q=None):
        q=self.q if q is None else q
        s1,c1=np.sin(q[0]),np.cos(q[0]); s12,c12=np.sin(q[0]+q[1]),np.cos(q[0]+q[1])
        return np.array([[-self.l1*s1-self.l2*s12, -self.l2*s12],[self.l1*c1+self.l2*c12, self.l2*c12]])
    def ik(self, target, q0=None, iters=80):
        q=self.q.copy() if q0 is None else np.array(q0,float)
        for _ in range(iters):
            e=target-self.fk(q)
            if np.linalg.norm(e)<1e-5: break
            q=q+0.6*(np.linalg.pinv(self.jac(q))@e)
        return q
    def step(self, tau):
        ddq=tau - self.damping*self.dq
        self.dq=self.dq+self.dt*ddq
        self.q=self.q+self.dt*self.dq
        return self.q.copy(), self.fk()

# smooth slow circle
T=400
ts=np.linspace(0,2*np.pi,T)
targets=np.stack([1.2*np.cos(ts), 1.2*np.sin(ts)],1)
# precompute IK path
arm0=TwoLinkArm(); q_path=np.zeros((T,2))
q=arm0.reset()
for i,tgt in enumerate(targets):
    q=arm0.ik(tgt,q0=q); q_path[i]=q
# finite-diff desired dq
qd_path=np.gradient(q_path, axis=0)/arm0.dt

def sim_pd(kp=80.0, kd=12.0, ff=1.0):
    arm=TwoLinkArm(); arm.reset(q_path[0])
    xs=[]; errs=[]; 
    for i in range(T):
        e=q_path[i]-arm.q
        de=qd_path[i]-arm.dq
        tau=kp*e + kd*de + ff*0.0  # pure PD + vel error
        arm.step(tau)
        xs.append(arm.fk()); errs.append(np.linalg.norm(arm.fk()-targets[i]))
    return np.array(xs), np.array(errs)

def sim_openloop():
    # hold first IK forever
    arm=TwoLinkArm(); arm.reset(q_path[0])
    xs=[]; errs=[]
    for i in range(T):
        xs.append(arm.fk()); errs.append(np.linalg.norm(arm.fk()-targets[i]))
        # no torque
        arm.step(np.zeros(2))
    return np.array(xs), np.array(errs)

t0=time.time()
xs_pd, err_pd = sim_pd(80,12)
xs_w, err_w = sim_pd(8,1.5)
xs_ol, err_ol = sim_openloop()
elapsed=time.time()-t0
print("PD",err_pd.mean(), err_pd[-50:].mean(), "weak",err_w.mean(), "open",err_ol.mean())


In [ ]:

fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].plot(targets[:,0],targets[:,1],"k--",alpha=0.5,label="ref")
axes[0].plot(xs_pd[:,0],xs_pd[:,1],label="PD")
axes[0].plot(xs_w[:,0],xs_w[:,1],alpha=0.8,label="weak PD")
axes[0].set_aspect("equal"); axes[0].legend(); axes[0].set_title("End-effector path")
axes[1].plot(err_pd,label="PD"); axes[1].plot(err_w,label="weak PD"); axes[1].plot(err_ol,label="open-loop")
axes[1].set_title("Tracking error"); axes[1].legend(); axes[1].set_xlabel("t")
fig.tight_layout(); fig.savefig(OUT/"stage06_arm_tracking.png", dpi=120); plt.close(fig)

fig,ax=plt.subplots(figsize=(4,4))
for idx in [0,80,160,240,320,399]:
    q=q_path[idx]
    p0=np.zeros(2); p1=np.array([np.cos(q[0]),np.sin(q[0])]); p2=p1+np.array([np.cos(q[0]+q[1]),np.sin(q[0]+q[1])])
    ax.plot([p0[0],p1[0],p2[0]],[p0[1],p1[1],p2[1]],"-o",alpha=0.6)
ax.plot(targets[:,0],targets[:,1],"k--",alpha=0.3); ax.set_aspect("equal"); ax.set_title("IK postures")
fig.tight_layout(); fig.savefig(OUT/"stage06_ik_postures.png", dpi=120); plt.close(fig)

payload={
  "ok": True,
  "stage":"06-classical-robotics",
  "title":"Grok-robotics-06-classical",
  "metrics":{
    "pd_mean_err": float(err_pd.mean()),
    "pd_final50_mean_err": float(err_pd[-50:].mean()),
    "weak_pd_mean_err": float(err_w.mean()),
    "openloop_mean_err": float(err_ol.mean()),
  },
  "gpu": gpu, "elapsed_sec": elapsed,
  "concept": "FK/IK + PD feedback for trajectory tracking",
  "new_capability": "classical robotics baseline before RL continuous control",
  "compare_to_previous": "Stages01-05 abstract MDPs; Stage06 embodiment geometry + feedback",
  "fix_note": "slower ref + velocity error PD, tuned gains",
}
assert payload["metrics"]["pd_mean_err"] < 0.25, payload
assert payload["metrics"]["pd_mean_err"] < payload["metrics"]["weak_pd_mean_err"] < payload["metrics"]["openloop_mean_err"], payload
(OUT/"results_stage06.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print("STAGE06_OK")
